# Do LLMs Play Nash? Strategic Behavior of Language Models
**CSCE 631 — Summer I 2026 | Texas A&M University**

This notebook implements the full pipeline:
1. Build the 12-game suite with known Nash equilibria
2. Query the TAMU LLM API at temperatures T = 0, 0.5, 1.0
3. Compute exploitability of LLM strategies relative to Nash
4. Visualize systematic biases


## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

from src.games import build_game_suite
from src.solvers import compute_nash_equilibria, compute_exploitability
from src.llm import TAMUClient
from src.llm.response_parser import parse_action_distribution, build_game_prompt
from src.analysis import plot_exploitability_heatmap, plot_strategy_comparison, plot_bias_summary

RESPONSES_DIR = Path('../data/responses')
TEMPERATURES = [0.0, 0.5, 1.0]
N_QUERIES = 5   # number of LLM queries per (game, temperature) cell

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Build Game Suite

In [ ]:
games = build_game_suite()
print(f'Loaded {len(games)} games:')
for g in games:
    print(f'  {g.name}  [{g.game_type}]  {g.num_row_actions}x{g.num_col_actions}')

## 2. Verify / Compute Nash Equilibria

In [ ]:
from src.solvers import compute_exploitability

game_ne = {}  # game_name -> list of (row_ne, col_ne)

for g in games:
    if g.known_ne:
        ne_list = g.known_ne
    else:
        ne_list = compute_nash_equilibria(g)
    game_ne[g.name] = ne_list

    # Sanity check: exploitability of all known NE should be ~0
    for (r, c) in ne_list:
        eps = compute_exploitability(g, r, c)
        status = 'OK' if eps < 1e-3 else f'WARN eps={eps:.4f}'
        print(f'{g.name:45s} NE exploitability={eps:.2e}  [{status}]')

## 3. Query LLM (TAMU API)

In [ ]:
# Initialize client — ensure .env has TAMU_API_COOKIE set
client = TAMUClient(model='gpt-4o', temperature=1.0)

# Cache structure: {game_name: {temperature: [response_str, ...]}}
cache_path = RESPONSES_DIR / 'responses_cache.json'
if cache_path.exists():
    with open(cache_path) as f:
        response_cache = json.load(f)
    print(f'Loaded cache with {sum(len(v) for v in response_cache.values())} game entries')
else:
    response_cache = {}

In [ ]:
for g in tqdm(games, desc='Games'):
    if g.name not in response_cache:
        response_cache[g.name] = {}

    prompt = build_game_prompt(
        g.name, g.row_actions, g.col_actions,
        g.row_payoffs, g.col_payoffs, player=1
    )

    for T in TEMPERATURES:
        key = str(T)
        if key not in response_cache[g.name]:
            response_cache[g.name][key] = []

        already = len(response_cache[g.name][key])
        needed = N_QUERIES - already
        if needed <= 0:
            continue

        for _ in range(needed):
            resp = client.query_game(prompt, temperature=T)
            response_cache[g.name][key].append(resp)

# Save cache
with open(cache_path, 'w') as f:
    json.dump(response_cache, f, indent=2)
print('Responses saved.')

## 4. Parse LLM Responses into Strategy Distributions

In [ ]:
# llm_strategies[game_name][temperature] = averaged probability distribution over row_actions
llm_strategies = {}

for g in games:
    llm_strategies[g.name] = {}
    for T in TEMPERATURES:
        key = str(T)
        responses = response_cache.get(g.name, {}).get(key, [])
        if not responses:
            llm_strategies[g.name][T] = np.ones(g.num_row_actions) / g.num_row_actions
            continue
        dists = [parse_action_distribution(r, g.row_actions) for r in responses]
        llm_strategies[g.name][T] = np.mean(dists, axis=0)

    print(f'{g.name}')
    for T in TEMPERATURES:
        dist = llm_strategies[g.name][T]
        print(f'  T={T}: {dict(zip(g.row_actions, dist.round(3)))}')

## 5. Compute Exploitability

In [ ]:
# For comparison col strategy, use the FIRST known NE col strategy (or uniform if none)
exploitability_results = {}  # {game_name: {temperature: exploitability}}

for g in games:
    exploitability_results[g.name] = {}
    ne_list = game_ne[g.name]
    # Use the Nash equilibrium col strategy as the opponent
    col_ne = ne_list[0][1] if ne_list else np.ones(g.num_col_actions) / g.num_col_actions

    for T in TEMPERATURES:
        row_strat = llm_strategies[g.name][T]
        eps = compute_exploitability(g, row_strat, col_ne)
        exploitability_results[g.name][T] = eps

# Print summary table
print(f'{'Game':<45} {'T=0.0':>8} {'T=0.5':>8} {'T=1.0':>8}')
print('-' * 72)
for g in games:
    row = f'{g.name:<45}'
    for T in TEMPERATURES:
        row += f" {exploitability_results[g.name][T]:>8.3f}"
    print(row)

## 6. Visualizations

In [ ]:
# 6a. Exploitability heatmap
fig, ax = plot_exploitability_heatmap(exploitability_results)
plt.show()

In [ ]:
# 6b. Per-game strategy comparison for select games
highlight_games = ["Prisoner's Dilemma (Classic)", "Matching Pennies",
                   "Battle of the Sexes", "Nash Bargaining (Symmetric Split-100)"]

for gname in highlight_games:
    g = next(x for x in games if x.name == gname)
    ne_row = game_ne[g.name][0][0] if game_ne[g.name] else np.ones(g.num_row_actions)/g.num_row_actions
    llm_dict = {T: llm_strategies[g.name][T] for T in TEMPERATURES}
    fig, ax = plot_strategy_comparison(g.name, g.row_actions, ne_row, llm_dict)
    plt.show()

In [ ]:
# 6c. Cooperation / deviation bias at T=0
# For PD variants: positive bias = LLM cooperates more than NE predicts (Defect=1.0)
bias_scores = {}
for g in games:
    ne_row = game_ne[g.name][0][0] if game_ne[g.name] else np.ones(g.num_row_actions)/g.num_row_actions
    llm_row = llm_strategies[g.name][0.0]
    # Signed deviation: positive = LLM plays action 0 more than NE
    bias_scores[g.name] = float(llm_row[0] - ne_row[0])

fig, ax = plot_bias_summary(bias_scores)
plt.show()

## 7. Analysis Summary

Fill in after running experiments. Key questions to answer:
- Which games show the highest exploitability? Why?
- Does temperature increase or decrease exploitability?
- Is there a cooperation bias in Prisoner's Dilemma variants?
- Does the LLM randomize correctly in zero-sum games (Matching Pennies, RPS)?
- Does the LLM use equality bias (50/50) in asymmetric bargaining?
